In [0]:
%pip install tensorflow

In [0]:
# Importing libraries

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col, count
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from sklearn.preprocessing import StandardScaler
import mlflow

def ensure_catalog():
    spark.sql("USE CATALOG workspace")
    spark.sql("USE DATABASE ml_layer")

ensure_catalog()

# load tables 
 
trans_train = spark.table("workspace.ml_layer.transaction_train_features")
trans_test  = spark.table("workspace.ml_layer.transaction_test_features")

print("Sampling datasets...")


fraud_train = trans_train.filter("is_fraud = 1").sample(fraction=0.80, seed=42)


legit_train = trans_train.filter("is_fraud = 0").sample(fraction=0.05, seed=42)

train_sample = fraud_train.union(legit_train)


test_sample = trans_test.sample(fraction=0.05, seed=42)

print(f"Train sample: {train_sample.count():,} rows")
print(f"Test sample:  {test_sample.count():,} rows")

train_fraud_rate = train_sample.filter("is_fraud=1").count() / train_sample.count()
test_fraud_rate  = test_sample.filter("is_fraud=1").count() / test_sample.count()

print(f"Train fraud rate: {train_fraud_rate:.2%}")
print(f"Test fraud rate:    {test_fraud_rate:.2%}")

# Extraction and normalization

def extract_arrays(df):
    df_arr = df.withColumn("features_arr", vector_to_array("features"))
    pdf    = df_arr.select("customer_id", "features_arr", "is_fraud").toPandas()
    X      = np.array(pdf["features_arr"].tolist())
    y      = pdf["is_fraud"].values.astype(float)
    ids    = pdf["customer_id"].values
    return X, y, ids

print("Extracting arrays...")
X_train, y_train, ids_train = extract_arrays(train_sample)
X_test,  y_test,  ids_test  = extract_arrays(test_sample)

# Normalization
scaler = StandardScaler()
scaler.fit(X_train[y_train == 0])
X_train = scaler.transform(X_train)
X_test  = scaler.transform(X_test)

n_features = X_train.shape[1]
print(f"Features per transaction: {n_features}")

# Building sequences

SEQ_LEN = 5   

def build_sequences_fast(X, y, customer_ids, seq_len=SEQ_LEN):
    """
    Vectorized sequence building — takes the LAST seq_len transactions
    per customer rather than all sliding windows.
    This reduces sequence count from millions to one per customer.
    """
    sequences, labels = [], []
    unique_customers  = np.unique(customer_ids)

    for cust_id in unique_customers:
        mask   = customer_ids == cust_id
        cust_X = X[mask]
        cust_y = y[mask]

        if len(cust_X) == 0:
            continue

        # Pad if fewer than seq_len transactions
        if len(cust_X) < seq_len:
            pad    = np.zeros((seq_len - len(cust_X), n_features))
            cust_X = np.vstack([pad, cust_X])
            cust_y = np.concatenate([np.zeros(seq_len - len(cust_y)), cust_y])


        sequences.append(cust_X[-seq_len:])
        labels.append(cust_y[-1])   # label of most recent transaction

    return np.array(sequences), np.array(labels)

print("Building sequences...")
X_seq_train, y_seq_train = build_sequences_fast(X_train, y_train, ids_train)
X_seq_test,  y_seq_test  = build_sequences_fast(X_test,  y_test,  ids_test)

print(f"Train sequences: {X_seq_train.shape}")
print(f"Test sequences:  {X_seq_test.shape}")
print(f"Train fraud rate in sequences: {y_seq_train.mean():.2%}")

# Building LSTM

def build_lstm(seq_len, n_features):
    inputs = keras.Input(shape=(seq_len, n_features), name="sequence_input")

    # Single LSTM layer
    x = layers.LSTM(32, return_sequences=False, name="lstm_1")(inputs)
    x = layers.Dropout(0.15)(x)
    x = layers.Dense(16, activation="relu")(x)
    x = layers.Dropout(0.1)(x)
    output = layers.Dense(1, activation="sigmoid", name="fraud_score")(x)

    model = keras.Model(inputs, output, name="LSTM_FraudDetector")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc", curve="PR")]  # PR-AUC directly
    )
    return model

# Training

fraud_count  = int(y_seq_train.sum())
legit_count  = int(len(y_seq_train) - fraud_count)

class_weight = {0: 1.0, 1: min(10.0, np.sqrt(legit_count / fraud_count))}
print(f"Class weight for fraud: {class_weight[1]:.1f}x")
print(f"Training on {len(X_seq_train):,} sequences")

lstm_model = build_lstm(SEQ_LEN, n_features)
lstm_model.summary()

username = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{username}/lstm")

with mlflow.start_run(run_name="LSTM_transactions_optimized"):
    history = lstm_model.fit(
        X_seq_train, y_seq_train,
        epochs=10,               # reduced from 20 — enough to see convergence
        batch_size=1024,         # larger batch = faster epochs on CPU
        validation_split=0.1,
        class_weight=class_weight,
        callbacks=[
            keras.callbacks.EarlyStopping(
                monitor="val_auc",
                patience=3,          # reduced from 4
                restore_best_weights=True,
                mode="max"),
            keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=1,
                min_lr=0.0001)
        ],
        verbose=1
    )

    # Evaluate
    y_proba_lstm = lstm_model.predict(
        X_seq_test,
        batch_size=2048,         
        verbose=1
    ).flatten()

    roc_auc = roc_auc_score(y_seq_test, y_proba_lstm)
    pr_auc  = average_precision_score(y_seq_test, y_proba_lstm)

    from sklearn.metrics import precision_recall_curve
    precisions, recalls, thresholds = precision_recall_curve(y_seq_test, y_proba_lstm)
    f1_scores      = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    best_idx       = f1_scores.argmax()
    best_threshold = float(thresholds[best_idx]) if best_idx < len(thresholds) else 0.5
    best_f1        = float(f1_scores[best_idx])

    y_pred_optimal = (y_proba_lstm >= best_threshold).astype(int)

    print(f"\n{'='*50}")
    print(f"  LSTM | Transactions (optimized)")
    print(f"{'='*50}")
    print(f"  ROC-AUC        : {roc_auc:.4f}")
    print(f"  PR-AUC         : {pr_auc:.4f}   ← key metric")
    print(f"  Best threshold : {best_threshold:.4f}")
    print(f"  F1 @ threshold : {best_f1:.4f}")
    print(f"\n{classification_report(y_seq_test, y_pred_optimal, target_names=['legit', 'fraud'])}")

    mlflow.log_metrics({
        "roc_auc":        roc_auc,
        "pr_auc":         pr_auc,
        "f1":             best_f1,
        "best_threshold": best_threshold
    })
    mlflow.log_param("seq_len",          SEQ_LEN)
    mlflow.log_param("lstm_units",       32)
    mlflow.log_param("epochs_run",       len(history.history["loss"]))
    mlflow.log_param("training_sample",  f"{len(X_seq_train):,} sequences")
    mlflow.log_param("sequence_strategy","last_window_per_customer")